# 34. The Hazard & IMDG Segregation Problem

## Tier 4: The AI/ML/RL Augmentation Method (Deep Q-Network Implementation)

### Goal
Implement Deep Reinforcement Learning to transform IMDG segregation into a sequential decision process where an agent learns optimal container placement policies through interaction with the maritime stowage environment.

### Key assumptions
- Container placement is treated as a sequential decision-making problem
- Agent learns through trial and error with reward feedback
- Neural network approximates Q-values for state-action pairs
- Experience replay and target networks stabilize training

### Approach (step-by-step)
1. Define the RL environment with state space, action space, and reward function
2. Implement Deep Q-Network with neural network approximation
3. Create training loop with experience replay and target network updates
4. Train agent on multiple episodes with epsilon-greedy exploration
5. Evaluate trained policy and analyze learning behavior
6. Compare performance with baseline methods

### What to look for in the results
- Learning curves showing reward improvement over episodes
- Success rate in finding segregation-compliant solutions
- Intelligent behaviors like prioritizing explosive containers for isolated positions
- Fast inference time for trained policy (milliseconds vs seconds)

### Concrete example (from the source)
Training the DQN agent on a 15-container scenario with 25 available positions across 3 holds. After 5000 training episodes, the agent learns to achieve 94% success rate in finding segregation-compliant solutions, compared to 23% random baseline. The trained policy demonstrates intelligent behaviors like prioritizing explosive containers for isolated positions and grouping compatible materials to maximize space efficiency.

In [1]:
# Import required packages for Deep Reinforcement Learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional, Deque
import random
import time
from collections import deque, namedtuple
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('default')
sns.set_palette("husl")

# For neural network implementation (using numpy for educational purposes)
# In production, you would typically use PyTorch or TensorFlow

In [ ]:
# Define data structures for DQN implementation
@dataclass
class Container:
    """Represents a container with dangerous goods"""
    id: str
    imdg_class: str
    weight: float = 20.0

@dataclass
class Position:
    """Represents a stowage position on the vessel"""
    id: str
    hold: int
    x: float
    y: float
    z: float
    available: bool = True

@dataclass
class SegregationRequirement:
    """Represents segregation requirements between container classes"""
    class1: str
    class2: str
    requirement: str
    min_distance: float
    different_hold: bool = False

# Experience tuple for replay buffer
Experience = namedtuple('Experience', 
                       ['state', 'action', 'reward', 'next_state', 'done'])

@dataclass
class DQNConfig:
    """DQN hyperparameters"""
    learning_rate: float = 0.001
    gamma: float = 0.95  # Discount factor
    epsilon_start: float = 1.0  # Initial exploration rate
    epsilon_end: float = 0.01   # Final exploration rate
    epsilon_decay: float = 0.995
    batch_size: int = 32
    memory_size: int = 10000
    target_update_freq: int = 100
    hidden_layers: List[int] = None
    
    def __post_init__(self):
        if self.hidden_layers is None:
            self.hidden_layers = [128, 64, 32]

In [ ]:
# Create the DQN example problem
def create_dqn_example():
    """Create a 15-container, 25-position problem for DQN training"""
    
    # Define 15 containers with diverse IMDG classes
    container_configs = [
        ("E1", "1.1"),   # Explosives (highest priority)
        ("E2", "1.4S"),  # Explosives
        ("F1", "3"),     # Flammable liquids
        ("F2", "3"),     # Flammable liquids
        ("F3", "3"),     # Flammable liquids
        ("C1", "8"),     # Corrosives
        ("C2", "8.1"),   # Corrosives
        ("G1", "2.1"),   # Flammable gases
        ("G2", "2.3"),   # Toxic gases
        ("O1", "5.1"),   # Organic peroxides
        ("T1", "6.1"),   # Toxic materials
        ("M1", "9"),     # Miscellaneous
        ("M2", "9"),     # Miscellaneous
        ("R1", "4.1"),   # Radioactive
        ("R2", "4.2"),   # Radioactive
    ]
    
    containers = [Container(cid, imdg_class, 20.0) for cid, imdg_class in container_configs]
    
    # Define 25 positions across 3 holds (9, 8, 8 positions)
    positions = []
    hold_distribution = [9, 8, 8]  # Positions per hold
    
    for hold_idx, num_positions in enumerate(hold_distribution):
        hold_num = hold_idx + 1
        for pos_idx in range(num_positions):
            x = pos_idx * 2.5  # 2.5m spacing
            positions.append(Position(f"H{hold_num}_P{pos_idx+1}", hold_num, x, 0, 0))
    
    # Define comprehensive segregation requirements
    segregation_requirements = [
        # Explosives (most restrictive)
        SegregationRequirement("1.1", "1.4S", "Away From", 5.0),
        SegregationRequirement("1.1", "3", "Away From", 6.0),
        SegregationRequirement("1.1", "8", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "2.1", "Away From", 6.0),
        SegregationRequirement("1.1", "2.3", "Away From", 6.0),
        SegregationRequirement("1.1", "5.1", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "6.1", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "4.1", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "4.2", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "9", "Away From", 3.0),
        
        # Class 1.4S explosives
        SegregationRequirement("1.4S", "3", "Away From", 3.0),
        SegregationRequirement("1.4S", "8", "Separated From", 0.0, True),
        SegregationRequirement("1.4S", "2.1", "Away From", 3.0),
        SegregationRequirement("1.4S", "2.3", "Away From", 3.0),
        SegregationRequirement("1.4S", "4.1", "Away From", 5.0),
        SegregationRequirement("1.4S", "4.2", "Away From", 5.0),
        
        # Flammable liquids and gases
        SegregationRequirement("3", "8", "Away From", 3.0),
        SegregationRequirement("3", "2.1", "No restriction", 0.0),
        SegregationRequirement("3", "2.3", "Away From", 3.0),
        SegregationRequirement("3", "5.1", "Away From", 3.0),
        SegregationRequirement("3", "6.1", "Away From", 3.0),
        SegregationRequirement("3", "9", "No restriction", 0.0),
        
        SegregationRequirement("2.1", "8", "Away From", 3.0),
        SegregationRequirement("2.1", "2.3", "Away From", 3.0),
        SegregationRequirement("2.1", "5.1", "Away From", 3.0),
        SegregationRequirement("2.1", "6.1", "Away From", 3.0),
        SegregationRequirement("2.1", "9", "No restriction", 0.0),
        
        # Corrosives
        SegregationRequirement("8", "2.3", "Away From", 3.0),
        SegregationRequirement("8", "5.1", "Away From", 3.0),
        SegregationRequirement("8", "6.1", "Away From", 3.0),
        SegregationRequirement("8", "9", "Away From", 3.0),
        
        # Other restrictions
        SegregationRequirement("5.1", "6.1", "Away From", 3.0),
        SegregationRequirement("5.1", "9", "Away From", 3.0),
        SegregationRequirement("6.1", "9", "Away From", 3.0),
        
        # Radioactive materials
        SegregationRequirement("4.1", "3", "Away From", 6.0),
        SegregationRequirement("4.1", "8", "Separated From", 0.0, True),
        SegregationRequirement("4.1", "9", "Away From", 3.0),
        SegregationRequirement("4.2", "3", "Away From", 6.0),
        SegregationRequirement("4.2", "8", "Separated From", 0.0, True),
        SegregationRequirement("4.2", "9", "Away From", 3.0),
    ]
    
    return containers, positions, segregation_requirements

# Create the DQN example
containers, positions, seg_reqs = create_dqn_example()

print(f"DQN Training Environment:")
print(f"  Containers: {len(containers)}")
print(f"  Positions: {len(positions)}")
print(f"  Segregation requirements: {len(seg_reqs)}")

print(f"\nContainer Distribution by Class:")
class_counts = {}
for c in containers:
    class_counts[c.imdg_class] = class_counts.get(c.imdg_class, 0) + 1
for imdg_class, count in sorted(class_counts.items()):
    print(f"  Class {imdg_class}: {count} containers")

print(f"\nPosition Distribution:")
hold_counts = {}
for p in positions:
    hold_counts[p.hold] = hold_counts.get(p.hold, 0) + 1
for hold in sorted(hold_counts.keys()):
    print(f"  Hold {hold}: {hold_counts[hold]} positions")

In [ ]:
# Utility functions for DQN environment
def calculate_distance_matrix(positions: List[Position]) -> Dict[Tuple[str, str], float]:
    """Calculate Euclidean distances between all position pairs"""
    distances = {}
    for i, pos1 in enumerate(positions):
        for j, pos2 in enumerate(positions):
            if i != j:
                dist = np.sqrt((pos1.x - pos2.x)**2 + (pos1.y - pos2.y)**2 + (pos1.z - pos2.z)**2)
                distances[(pos1.id, pos2.id)] = dist
    return distances

def check_segregation_compliance(container1: Container, container2: Container, 
                                pos1: Position, pos2: Position, 
                                requirements: List[SegregationRequirement],
                                distances: Dict[Tuple[str, str], float]) -> bool:
    """Check if two containers in given positions satisfy segregation requirements"""
    
    # Find applicable requirement
    applicable_req = None
    for req in requirements:
        if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
            (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
            applicable_req = req
            break
    
    if applicable_req is None or applicable_req.requirement == "No restriction":
        return True
    
    # Check distance requirement
    distance = distances.get((pos1.id, pos2.id), 0)
    if distance < applicable_req.min_distance:
        return False
    
    # Check different hold requirement
    if applicable_req.different_hold and pos1.hold == pos2.hold:
        return False
    
    return True

# Calculate distance matrix
distance_matrix = calculate_distance_matrix(positions)

In [ ]:
# Neural Network implementation (simplified for educational purposes)
class NeuralNetwork:
    """Simple neural network for DQN using numpy"""
    
    def __init__(self, input_size: int, output_size: int, hidden_layers: List[int], learning_rate: float = 0.001):
        self.input_size = input_size
        self.output_size = output_size
        self.learning_rate = learning_rate
        
        # Initialize network architecture
        layer_sizes = [input_size] + hidden_layers + [output_size]
        
        # Initialize weights and biases
        self.weights = []
        self.biases = []
        
        for i in range(len(layer_sizes) - 1):
            # Xavier initialization
            limit = np.sqrt(6 / (layer_sizes[i] + layer_sizes[i + 1]))
            self.weights.append(np.random.uniform(-limit, limit, (layer_sizes[i], layer_sizes[i + 1])))
            self.biases.append(np.zeros(layer_sizes[i + 1]))
        
        # Store for backpropagation
        self.activations = []
        self.z_values = []
    
    def relu(self, x: np.ndarray) -> np.ndarray:
        """ReLU activation function"""
        return np.maximum(0, x)
    
    def relu_derivative(self, x: np.ndarray) -> np.ndarray:
        """ReLU derivative"""
        return (x > 0).astype(float)
    
    def forward(self, x: np.ndarray) -> np.ndarray:
        """Forward propagation"""
        self.activations = [x]
        self.z_values = []
        
        current = x
        
        # Hidden layers with ReLU
        for i in range(len(self.weights) - 1):
            z = np.dot(current, self.weights[i]) + self.biases[i]
            self.z_values.append(z)
            current = self.relu(z)
            self.activations.append(current)
        
        # Output layer (linear)
        z = np.dot(current, self.weights[-1]) + self.biases[-1]
        self.z_values.append(z)
        self.activations.append(z)
        
        return z
    
    def backward(self, target: np.ndarray, predicted: np.ndarray):
        """Backward propagation with gradient descent"""
        # Calculate output layer gradient
        delta = (predicted - target)  # MSE loss gradient
        
        # Backpropagate through layers
        for i in range(len(self.weights) - 1, -1, -1):
            # Get activation for this layer
            if i == len(self.weights) - 1:
                # Output layer (linear activation)
                activation_grad = 1.0
            else:
                # Hidden layer (ReLU activation)
                activation_grad = self.relu_derivative(self.z_values[i])
            
            # Calculate gradients
            weight_grad = np.outer(self.activations[i], delta * activation_grad)
            bias_grad = delta * activation_grad
            
            # Update weights and biases
            self.weights[i] -= self.learning_rate * weight_grad
            self.biases[i] -= self.learning_rate * bias_grad
            
            # Propagate delta to previous layer
            if i > 0:
                delta = np.dot(delta * activation_grad, self.weights[i].T)
    
    def copy(self) -> 'NeuralNetwork':
        """Create a copy of the neural network"""
        new_net = NeuralNetwork(self.input_size, self.output_size, 
                              [w.shape[1] for w in self.weights[:-1]], self.learning_rate)
        new_net.weights = [w.copy() for w in self.weights]
        new_net.biases = [b.copy() for b in self.biases]
        return new_net
    
    def update_weights(self, other_network: 'NeuralNetwork', tau: float = 0.005):
        """Soft update of network weights"""
        for i in range(len(self.weights)):
            self.weights[i] = (1 - tau) * self.weights[i] + tau * other_network.weights[i]
            self.biases[i] = (1 - tau) * self.biases[i] + tau * other_network.biases[i]

In [ ]:
# DQN Environment implementation
class IMDGEnvironment:
    """Custom environment for IMDG segregation problem"""
    
    def __init__(self, containers: List[Container], positions: List[Position], 
                 requirements: List[SegregationRequirement], distances: Dict[Tuple[str, str], float]):
        self.containers = containers
        self.positions = positions
        self.requirements = requirements
        self.distances = distances
        
        self.reset()
    
    def reset(self) -> np.ndarray:
        """Reset environment to initial state"""
        self.current_step = 0
        self.available_positions = [p.id for p in self.positions]
        self.placed_containers = []  # List of (container, position) tuples
        self.current_violations = 0
        
        return self._get_state()
    
    def _get_state(self) -> np.ndarray:
        """Get current state representation"""
        # State features:
        # 1. Current step (normalized)
        # 2. Number of available positions (normalized)
        # 3. Current violation count (normalized)
        # 4-7. One-hot encoding of current container IMDG class
        # 8-15. Availability of each position (binary)
        # 16-31. Hold distribution of placed containers (8 holds * 2 features)
        
        state = np.zeros(32)
        
        # Basic state info
        state[0] = self.current_step / len(self.containers)
        state[1] = len(self.available_positions) / len(self.positions)
        state[2] = min(self.current_violations / 10.0, 1.0)  # Cap at 10 violations
        
        # Current container IMDG class (one-hot)
        if self.current_step < len(self.containers):
            current_container = self.containers[self.current_step]
            imdg_classes = ["1.1", "1.4S", "3", "8", "2.1", "2.3", "5.1", "6.1", "4.1", "4.2", "9"]
            if current_container.imdg_class in imdg_classes:
                class_idx = imdg_classes.index(current_container.imdg_class)
                if class_idx < 4:  # Only first 4 classes due to state size limit
                    state[3 + class_idx] = 1.0
        
        # Position availability (first 8 positions)
        for i, pos_id in enumerate(self.positions[:8]):
            if pos_id.id in self.available_positions:
                state[8 + i] = 1.0
        
        # Hold distribution (8 holds, 2 features each)
        hold_counts = {}
        for _, pos in self.placed_containers:
            hold_counts[pos.hold] = hold_counts.get(pos.hold, 0) + 1
        
        for hold in range(1, 9):  # Holds 1-8
            count = hold_counts.get(hold, 0)
            state[16 + (hold-1)*2] = count / 5.0  # Normalized count
            state[16 + (hold-1)*2 + 1] = 1.0 if count > 0 else 0.0  # Binary indicator
        
        return state
    
    def get_valid_actions(self) -> List[int]:
        """Get list of valid actions (available position indices)"""
        valid_actions = []
        for i, pos in enumerate(self.positions):
            if pos.id in self.available_positions:
                valid_actions.append(i)
        return valid_actions
    
    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict]:
        """Execute action and return (next_state, reward, done, info)"""
        
        if self.current_step >= len(self.containers):
            return self._get_state(), 0, True, {"error": "Episode already finished"}
        
        if action >= len(self.positions) or self.positions[action].id not in self.available_positions:
            # Invalid action
            return self._get_state(), -10, False, {"error": "Invalid action"}
        
        # Place container
        current_container = self.containers[self.current_step]
        chosen_position = self.positions[action]
        
        # Calculate reward
        reward = self._calculate_reward(current_container, chosen_position)
        
        # Update state
        self.placed_containers.append((current_container, chosen_position))
        self.available_positions.remove(chosen_position.id)
        self.current_step += 1
        
        # Recalculate violations
        self._update_violations()
        
        # Check if episode is done
        done = self.current_step >= len(self.containers)
        
        # Bonus for completing successfully
        if done and self.current_violations == 0:
            reward += 100
        elif done:
            reward -= 50 * self.current_violations
        
        info = {
            "container_id": current_container.id,
            "position_id": chosen_position.id,
            "violations": self.current_violations,
            "step": self.current_step
        }
        
        return self._get_state(), reward, done, info
    
    def _calculate_reward(self, container: Container, position: Position) -> float:
        """Calculate reward for placing container at position"""
        reward = -1  # Base time penalty
        
        # Check for new violations
        new_violations = 0
        for placed_container, placed_position in self.placed_containers:
            if not check_segregation_compliance(container, placed_container, position, placed_position, 
                                             self.requirements, self.distances):
                new_violations += 1
                
                # Penalty based on violation severity
                for req in self.requirements:
                    if ((req.class1 == container.imdg_class and req.class2 == placed_container.imdg_class) or
                        (req.class1 == placed_container.imdg_class and req.class2 == container.imdg_class)):
                        if req.different_hold and position.hold == placed_position.hold:
                            reward -= 50  # Same hold violation
                        else:
                            reward -= 20  # Distance violation
                        break
        
        # Reward for good placement
        if new_violations == 0:
            # Bonus for placing high-risk containers in good positions
            if container.imdg_class in ["1.1", "1.4S"]:
                if position.hold == 1 and len(self.placed_containers) < 3:  # Early placement in first hold
                    reward += 10
            elif container.imdg_class in ["8", "8.1"]:
                if position.hold != 1:  # Corrosives away from explosives
                    reward += 5
        
        return reward
    
    def _update_violations(self):
        """Update current violation count"""
        self.current_violations = 0
        for i, (container1, pos1) in enumerate(self.placed_containers):
            for j, (container2, pos2) in enumerate(self.placed_containers):
                if i < j:
                    if not check_segregation_compliance(container1, container2, pos1, pos2, 
                                                     self.requirements, self.distances):
                        self.current_violations += 1

In [ ]:
# DQN Agent implementation
class DQNAgent:
    """Deep Q-Network Agent for IMDG segregation"""
    
    def __init__(self, state_size: int, action_size: int, config: DQNConfig):
        self.state_size = state_size
        self.action_size = action_size
        self.config = config
        
        # Initialize neural networks
        self.q_network = NeuralNetwork(state_size, action_size, config.hidden_layers, config.learning_rate)
        self.target_network = self.q_network.copy()
        
        # Experience replay buffer
        self.memory = deque(maxlen=config.memory_size)
        
        # Training variables
        self.epsilon = config.epsilon_start
        self.training_step = 0
    
    def act(self, state: np.ndarray, valid_actions: List[int], training: bool = True) -> int:
        """Choose action using epsilon-greedy policy"""
        if training and random.random() < self.epsilon:
            # Explore: choose random valid action
            return random.choice(valid_actions) if valid_actions else 0
        else:
            # Exploit: choose best valid action
            if valid_actions:
                q_values = self.q_network.forward(state)
                # Mask invalid actions
                masked_q = np.full(self.action_size, -np.inf)
                masked_q[valid_actions] = q_values[valid_actions]
                return np.argmax(masked_q)
            return 0
    
    def remember(self, state: np.ndarray, action: int, reward: float, 
                next_state: np.ndarray, done: bool):
        """Store experience in replay buffer"""
        experience = Experience(state, action, reward, next_state, done)
        self.memory.append(experience)
    
    def replay(self):
        """Train the model using experience replay"""
        if len(self.memory) < self.config.batch_size:
            return
        
        # Sample random batch from memory
        batch = random.sample(self.memory, self.config.batch_size)
        
        # Prepare batch data
        states = np.array([e.state for e in batch])
        actions = np.array([e.action for e in batch])
        rewards = np.array([e.reward for e in batch])
        next_states = np.array([e.next_state for e in batch])
        dones = np.array([e.done for e in batch])
        
        # Current Q-values
        current_q_values = self.q_network.forward(states)
        
        # Next Q-values from target network
        next_q_values = self.target_network.forward(next_states)
        max_next_q = np.max(next_q_values, axis=1)
        
        # Target Q-values
        target_q = current_q_values.copy()
        for i in range(self.config.batch_size):
            if dones[i]:
                target_q[i, actions[i]] = rewards[i]
            else:
                target_q[i, actions[i]] = rewards[i] + self.config.gamma * max_next_q[i]
        
        # Train the network
        self.q_network.backward(target_q, current_q_values)
        
        # Update target network
        self.training_step += 1
        if self.training_step % self.config.target_update_freq == 0:
            self.target_network.update_weights(self.q_network)
    
    def update_epsilon(self):
        """Update epsilon for exploration"""
        self.epsilon = max(self.config.epsilon_end, self.epsilon * self.config.epsilon_decay)
    
    def save_model(self, filepath: str):
        """Save model weights"""
        model_data = {
            'weights': self.q_network.weights,
            'biases': self.q_network.biases,
            'config': self.config
        }
        np.save(filepath, model_data)
    
    def load_model(self, filepath: str):
        """Load model weights"""
        model_data = np.load(filepath, allow_pickle=True).item()
        self.q_network.weights = model_data['weights']
        self.q_network.biases = model_data['biases']
        self.target_network.weights = [w.copy() for w in self.q_network.weights]
        self.target_network.biases = [b.copy() for b in self.q_network.biases]

In [ ]:
# Training function
def train_dqn_agent(episodes: int = 1000, max_steps_per_episode: int = 20) -> Dict:
    """Train DQN agent on IMDG segregation problem"""
    
    # Initialize environment and agent
    env = IMDGEnvironment(containers, positions, seg_reqs, distance_matrix)
    
    config = DQNConfig(
        learning_rate=0.001,
        gamma=0.95,
        epsilon_start=1.0,
        epsilon_end=0.01,
        epsilon_decay=0.995,
        batch_size=32,
        memory_size=10000,
        target_update_freq=100,
        hidden_layers=[128, 64, 32]
    )
    
    agent = DQNAgent(state_size=32, action_size=len(positions), config=config)
    
    # Training metrics
    episode_rewards = []
    episode_violations = []
    episode_success_rates = []
    epsilon_history = []
    
    print(f"DQN Training - {episodes} episodes, max {max_steps_per_episode} steps per episode")
    print("=" * 70)
    
    for episode in range(episodes):
        state = env.reset()
        total_reward = 0
        steps = 0
        
        for step in range(max_steps_per_episode):
            # Get valid actions
            valid_actions = env.get_valid_actions()
            
            if not valid_actions:
                break
            
            # Choose action
            action = agent.act(state, valid_actions, training=True)
            
            # Execute action
            next_state, reward, done, info = env.step(action)
            
            # Store experience
            agent.remember(state, action, reward, next_state, done)
            
            # Update state
            state = next_state
            total_reward += reward
            steps += 1
            
            if done:
                break
        
        # Train agent
        agent.replay()
        agent.update_epsilon()
        
        # Record metrics
        episode_rewards.append(total_reward)
        episode_violations.append(env.current_violations)
        success = 1 if env.current_violations == 0 else 0
        episode_success_rates.append(success)
        epsilon_history.append(agent.epsilon)
        
        # Progress reporting
        if (episode + 1) % 100 == 0 or episode == 0:
            avg_reward = np.mean(episode_rewards[-100:]) if len(episode_rewards) >= 100 else np.mean(episode_rewards)
            avg_success = np.mean(episode_success_rates[-100:]) if len(episode_success_rates) >= 100 else np.mean(episode_success_rates)
            print(f"Episode {episode + 1:4d}: Reward = {total_reward:6.1f}, Avg Reward = {avg_reward:6.1f}, Success Rate = {avg_success:.3f}, Epsilon = {agent.epsilon:.3f}")
    
    return {
        'agent': agent,
        'episode_rewards': episode_rewards,
        'episode_violations': episode_violations,
        'episode_success_rates': episode_success_rates,
        'epsilon_history': epsilon_history,
        'final_success_rate': np.mean(episode_success_rates[-100:]) if len(episode_success_rates) >= 100 else np.mean(episode_success_rates)
    }

In [ ]:
# Train the DQN agent
print("IMDG Segregation - Deep Q-Network Training")
print("=" * 60)
print(f"Environment: {len(containers)} containers, {len(positions)} positions")
print(f"State size: 32 features, Action size: {len(positions)} possible positions")
print()

# Train agent (reduced episodes for demonstration)
training_result = train_dqn_agent(episodes=500, max_steps_per_episode=20)

print("\n" + "=" * 60)
print("DQN TRAINING RESULTS:")
print("=" * 60)
print(f"Final Success Rate: {training_result['final_success_rate']:.3f} ({training_result['final_success_rate']*100:.1f}%)")
print(f"Final Epsilon: {training_result['agent'].epsilon:.4f}")
print(f"Training Steps: {training_result['agent'].training_step}")

# Calculate improvement over random baseline
random_success_rate = 0.23  # From source: 23% random baseline
improvement = ((training_result['final_success_rate'] - random_success_rate) / random_success_rate) * 100

print(f"\nBaseline Comparison:")
print(f"Random Baseline Success Rate: {random_success_rate:.1%}")
print(f"DQN Success Rate: {training_result['final_success_rate']:.1%}")
print(f"Improvement: {improvement:.1f}%")

In [ ]:
# Evaluate trained agent
def evaluate_trained_agent(agent: DQNAgent, num_episodes: int = 50) -> Dict:
    """Evaluate trained agent performance"""
    
    env = IMDGEnvironment(containers, positions, seg_reqs, distance_matrix)
    
    evaluation_rewards = []
    evaluation_violations = []
    evaluation_success = []
    placement_history = []
    
    print(f"\nEvaluating trained agent for {num_episodes} episodes...")
    
    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0
        episode_placements = []
        
        for step in range(len(containers)):
            valid_actions = env.get_valid_actions()
            
            if not valid_actions:
                break
            
            # Choose action (no exploration during evaluation)
            action = agent.act(state, valid_actions, training=False)
            
            # Execute action
            next_state, reward, done, info = env.step(action)
            
            episode_placements.append({
                'step': step,
                'container_id': info['container_id'],
                'position_id': info['position_id'],
                'violations_after': info['violations']
            })
            
            state = next_state
            total_reward += reward
            
            if done:
                break
        
        evaluation_rewards.append(total_reward)
        evaluation_violations.append(env.current_violations)
        evaluation_success.append(1 if env.current_violations == 0 else 0)
        placement_history.append(episode_placements)
    
    return {
        'avg_reward': np.mean(evaluation_rewards),
        'avg_violations': np.mean(evaluation_violations),
        'success_rate': np.mean(evaluation_success),
        'placement_history': placement_history,
        'all_rewards': evaluation_rewards,
        'all_violations': evaluation_violations
    }

# Evaluate the trained agent
evaluation_result = evaluate_trained_agent(training_result['agent'], num_episodes=50)

print("\nEVALUATION RESULTS:")
print("=" * 40)
print(f"Average Reward: {evaluation_result['avg_reward']:.2f}")
print(f"Average Violations: {evaluation_result['avg_violations']:.2f}")
print(f"Success Rate: {evaluation_result['success_rate']:.3f} ({evaluation_result['success_rate']*100:.1f}%)")

In [ ]:
# Visualize DQN training and results
def visualize_dqn_results(training_result: Dict, evaluation_result: Dict):
    """Create comprehensive visualization of DQN results"""
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Training rewards over episodes
    ax1 = axes[0, 0]
    episodes = range(1, len(training_result['episode_rewards']) + 1)
    ax1.plot(episodes, training_result['episode_rewards'], 'b-', alpha=0.7, linewidth=1)
    
    # Moving average
    window = min(50, len(training_result['episode_rewards']) // 4)
    if window > 1:
        moving_avg = pd.Series(training_result['episode_rewards']).rolling(window).mean()
        ax1.plot(episodes, moving_avg, 'r-', linewidth=2, label=f'Moving Avg ({window} episodes)')
    
    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Total Reward')
    ax1.set_title('Training Progress - Rewards')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Success rate over episodes
    ax2 = axes[0, 1]
    success_rates = training_result['episode_success_rates']
    ax2.plot(episodes, success_rates, 'g-', alpha=0.7, linewidth=1)
    
    # Moving average for success rate
    if window > 1:
        success_moving_avg = pd.Series(success_rates).rolling(window).mean()
        ax2.plot(episodes, success_moving_avg, 'r-', linewidth=2, label=f'Moving Avg ({window} episodes)')
    
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Success Rate')
    ax2.set_title('Training Progress - Success Rate')
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Epsilon decay
    ax3 = axes[0, 2]
    ax3.plot(episodes, training_result['epsilon_history'], 'purple', linewidth=2)
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Epsilon (Exploration Rate)')
    ax3.set_title('Exploration Rate Decay')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Violations distribution
    ax4 = axes[1, 0]
    ax4.hist(training_result['episode_violations'], bins=range(max(training_result['episode_violations']) + 2), 
             alpha=0.7, color='orange', edgecolor='black')
    ax4.set_xlabel('Number of Violations')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Training - Violations Distribution')
    ax4.grid(True, alpha=0.3, axis='y')
    
    # Plot 5: Evaluation results comparison
    ax5 = axes[1, 1]
    methods = ['Random Baseline', 'DQN Trained']
    success_rates = [0.23, evaluation_result['success_rate']]
    colors = ['red', 'green']
    
    bars = ax5.bar(methods, success_rates, color=colors, alpha=0.7)
    ax5.set_ylabel('Success Rate')
    ax5.set_title('Performance Comparison')
    ax5.set_ylim(0, 1)
    ax5.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    for bar, rate in zip(bars, success_rates):
        ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{rate:.1%}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 6: Learning curve analysis
    ax6 = axes[1, 2]
    
    # Calculate success rate in different phases
    total_episodes = len(training_result['episode_success_rates'])
    phases = ['Early', 'Mid', 'Late']
    phase_ranges = [(0, total_episodes//3), 
                   (total_episodes//3, 2*total_episodes//3),
                   (2*total_episodes//3, total_episodes)]
    
    phase_success_rates = []
    for start, end in phase_ranges:
        if end > start:
            phase_rate = np.mean(training_result['episode_success_rates'][start:end])
            phase_success_rates.append(phase_rate)
        else:
            phase_success_rates.append(0)
    
    bars2 = ax6.bar(phases, phase_success_rates, color=['lightblue', 'blue', 'darkblue'], alpha=0.7)
    ax6.set_ylabel('Success Rate')
    ax6.set_title('Learning Progress by Phase')
    ax6.set_ylim(0, 1)
    ax6.grid(True, alpha=0.3, axis='y')
    
    # Add percentage labels
    for bar, rate in zip(bars2, phase_success_rates):
        ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
                f'{rate:.1%}', ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('IMDG Segregation - DQN Training Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Analyze placement patterns
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Extract placement patterns from successful episodes
    successful_placements = []
    for i, violations in enumerate(evaluation_result['all_violations']):
        if violations == 0 and i < len(evaluation_result['placement_history']):
            successful_placements.extend(evaluation_result['placement_history'][i])
    
    if successful_placements:
        # Container placement order analysis
        placement_df = pd.DataFrame(successful_placements)
        
        # Count placements by step
        step_counts = placement_df['step'].value_counts().sort_index()
        ax1.bar(step_counts.index, step_counts.values, color='skyblue', alpha=0.7)
        ax1.set_xlabel('Placement Step')
        ax1.set_ylabel('Frequency')
        ax1.set_title('Container Placement Order in Successful Episodes')
        ax1.grid(True, alpha=0.3, axis='y')
        
        # Hold preference analysis
        hold_data = []
        for _, placement in enumerate(successful_placements):
            position_id = placement['position_id']
            hold_num = int(position_id.split('_')[0][1:])  # Extract hold number from "H1_P1"
            hold_data.append(hold_num)
        
        hold_counts = pd.Series(hold_data).value_counts().sort_index()
        ax2.bar(hold_counts.index, hold_counts.values, color='lightgreen', alpha=0.7)
        ax2.set_xlabel('Hold Number')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Hold Preference in Successful Episodes')
        ax2.grid(True, alpha=0.3, axis='y')
        
        plt.suptitle('DQN Agent - Placement Pattern Analysis', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Visualize DQN results
visualize_dqn_results(training_result, evaluation_result)

In [ ]:
# Policy analysis and inference speed test
def analyze_trained_policy(agent: DQNAgent):
    """Analyze the learned policy and test inference speed"""
    
    print("\nPOLICY ANALYSIS:")
    print("=" * 40)
    
    # Test inference speed
    env = IMDGEnvironment(containers, positions, seg_reqs, distance_matrix)
    state = env.reset()
    
    # Measure inference time
    num_tests = 1000
    start_time = time.time()
    
    for _ in range(num_tests):
        valid_actions = env.get_valid_actions()
        if valid_actions:
            action = agent.act(state, valid_actions, training=False)
    
    inference_time = (time.time() - start_time) / num_tests * 1000  # Convert to milliseconds
    
    print(f"Inference Speed: {inference_time:.3f} ms per decision")
    print(f"Decisions per second: {1000 / inference_time:.0f}")
    
    # Analyze Q-value distribution
    q_values = agent.q_network.forward(state)
    
    print(f"\nQ-Value Statistics:")
    print(f"  Mean Q-value: {np.mean(q_values):.3f}")
    print(f"  Std Q-value: {np.std(q_values):.3f}")
    print(f"  Min Q-value: {np.min(q_values):.3f}")
    print(f"  Max Q-value: {np.max(q_values):.3f}")
    
    # Test policy on a few example states
    print(f"\nExample Policy Decisions:")
    for test_case in range(3):
        state = env.reset()
        valid_actions = env.get_valid_actions()
        
        if valid_actions:
            action = agent.act(state, valid_actions, training=False)
            q_vals = agent.q_network.forward(state)
            
            print(f"\nTest Case {test_case + 1}:")
            print(f"  Valid actions: {len(valid_actions)} available")
            print(f"  Chosen action: {action} -> Position {positions[action].id} (Hold {positions[action].hold})")
            print(f"  Q-value of chosen action: {q_vals[action]:.3f}")
            print(f"  Average Q-value of valid actions: {np.mean(q_vals[valid_actions]):.3f}")
            
            # Execute action to see result
            next_state, reward, done, info = env.step(action)
            print(f"  Immediate reward: {reward:.1f}")
            print(f"  Container placed: {info['container_id']} (Class {containers[test_case].imdg_class})")
            
            # Check if this was a good placement
            if test_case < len(containers):
                container = containers[test_case]
                if container.imdg_class in ["1.1", "1.4S"] and positions[action].hold == 1:
                    print(f"  ✓ Good placement: Explosive in isolated hold 1")
                elif container.imdg_class in ["8", "8.1"] and positions[action].hold != 1:
                    print(f"  ✓ Good placement: Corrosive away from explosive hold")
                else:
                    print(f"  ○ Standard placement")

# Analyze the trained policy
analyze_trained_policy(training_result['agent'])

### Why this Tier exists vs earlier Tiers
This Tier 4 Deep Reinforcement Learning approach addresses limitations of previous methods:
- **Adaptive decision-making** - Learns optimal policies from experience rather than following fixed rules
- **Sequential optimization** - Handles the sequential nature of container placement naturally
- **Experience-based learning** - Improves performance through interaction with the environment
- **Fast inference** - Once trained, makes decisions in milliseconds vs seconds for optimization
- **Generalization** - Can handle new problem instances without reprogramming

### Pros / Cons vs Tiers 1-3
**Advantages over all previous Tiers:**
- **Fast inference time** - Milliseconds vs seconds/minutes for optimization
- **Adaptive behavior** - Learns from experience and adapts to new scenarios
- **Sequential decision-making** - Naturally handles the order-dependent nature of placement
- **Scalability after training** - Can solve new instances quickly once trained
- **Continuous improvement** - Performance improves with more training data

**Disadvantages vs Tier 1 (MIP):**
- No optimality guarantees
- Requires extensive training
- Performance depends on training quality
- May not find truly optimal solutions

**Disadvantages vs Tier 2 (Heuristics):**
- Much more complex to implement and train
- Requires large amounts of training data
- Less interpretable decision-making
- Higher computational cost for training

**Disadvantages vs Tier 3 (PSO):**
- Training is much more computationally expensive
- More hyperparameters to tune
- Less transparent optimization process
- Requires careful environment design

### When to use this Tier
- **Real-time operations** requiring millisecond decision times
- **Dynamic environments** with frequently changing constraints
- **Large-scale deployment** where the same problem type occurs repeatedly
- **Integration with automated systems** requiring rapid decision-making
- **When inference speed is more critical than solution optimality**
- **Applications with similar problem instances** where training investment pays off
- **Systems requiring adaptive behavior** that improves over time